#  ISL Continuous Sign Language Translator

**State-of-the-art Indian Sign Language Recognition & Translation**

This notebook trains a pose-based ISL translator on Google Colab.

## Features
- MediaPipe Holistic (543 keypoints)
- ST-GCN + Transformer architecture
- CTC-based gloss recognition
- IndicTrans2 for Hindi/English translation

## 1. Setup Environment

In [ ]:
# Check GPU
!nvidia-smi

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")

In [ ]:
# Install dependencies
!pip install -q mediapipe opencv-python transformers sentencepiece
!pip install -q tqdm tensorboard pyyaml omegaconf
!pip install -q jiwer sacrebleu editdistance
!pip install -q kaggle rich

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Set project directory
PROJECT_DIR = '/content/drive/MyDrive/isl-translator'
!mkdir -p {PROJECT_DIR}

In [ ]:
# Clone or copy project files
# Option 1: If you have it on GitHub
# !git clone https://github.com/YOUR_USERNAME/isl-translator.git

# Option 2: Upload from local (run this cell and upload the zip)
from google.colab import files
# uploaded = files.upload()  # Uncomment to upload

# Option 3: Copy from Drive
import sys
sys.path.insert(0, PROJECT_DIR)

## 2. Download Dataset

In [ ]:
# Setup Kaggle credentials
# Upload your kaggle.json file
!mkdir -p ~/.kaggle

# Uncomment and run this to upload kaggle.json
# from google.colab import files
# uploaded = files.upload()
# !mv kaggle.json ~/.kaggle/
# !chmod 600 ~/.kaggle/kaggle.json

In [ ]:
# Download ISL-CSLTR dataset from Kaggle
# Search for the exact dataset name on Kaggle first
DATA_DIR = f"{PROJECT_DIR}/data/raw"
!mkdir -p {DATA_DIR}

# Example (update with actual dataset name):
# !kaggle datasets download -d username/isl-csltr -p {DATA_DIR} --unzip

## 3. Extract Keypoints

In [ ]:
import cv2
import numpy as np
import mediapipe as mp
from pathlib import Path
from tqdm.notebook import tqdm
import json

class KeypointExtractor:
    """Extract MediaPipe Holistic keypoints."""
    
    def __init__(self):
        self.mp_holistic = mp.solutions.holistic
        self.holistic = self.mp_holistic.Holistic(
            static_image_mode=False,
            model_complexity=2,
            min_detection_confidence=0.5,
            min_tracking_confidence=0.5,
        )
    
    def extract_video(self, video_path):
        cap = cv2.VideoCapture(video_path)
        keypoints_list = []
        
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = self.holistic.process(rgb)
            
            kp = np.zeros((543, 3), dtype=np.float32)
            idx = 0
            
            if results.pose_landmarks:
                for lm in results.pose_landmarks.landmark:
                    kp[idx] = [lm.x, lm.y, lm.visibility]
                    idx += 1
            else:
                idx += 33
            
            if results.face_landmarks:
                for lm in results.face_landmarks.landmark:
                    kp[idx] = [lm.x, lm.y, lm.z]
                    idx += 1
            else:
                idx += 468
            
            if results.left_hand_landmarks:
                for lm in results.left_hand_landmarks.landmark:
                    kp[idx] = [lm.x, lm.y, lm.z]
                    idx += 1
            else:
                idx += 21
            
            if results.right_hand_landmarks:
                for lm in results.right_hand_landmarks.landmark:
                    kp[idx] = [lm.x, lm.y, lm.z]
                    idx += 1
            
            keypoints_list.append(kp)
        
        cap.release()
        return np.array(keypoints_list)

print("KeypointExtractor ready!")

In [ ]:
# Extract keypoints from dataset
PROCESSED_DIR = f"{PROJECT_DIR}/data/processed"
!mkdir -p {PROCESSED_DIR}/train {PROCESSED_DIR}/val {PROCESSED_DIR}/test

extractor = KeypointExtractor()

# Find all videos
video_dir = Path(DATA_DIR)
videos = list(video_dir.glob('**/*.mp4')) + list(video_dir.glob('**/*.avi'))
print(f"Found {len(videos)} videos")

# Extract (this takes a while)
for video_path in tqdm(videos[:10]):  # Start with first 10 for testing
    output_path = f"{PROCESSED_DIR}/train/{video_path.stem}.npz"
    
    if Path(output_path).exists():
        continue
    
    try:
        kp = extractor.extract_video(str(video_path))
        np.savez_compressed(output_path, keypoints=kp)
        print(f"✓ {video_path.name}: {kp.shape}")
    except Exception as e:
        print(f"✗ {video_path.name}: {e}")

## 4. Create Model

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))
    
    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)


class ISLTranslator(nn.Module):
    def __init__(self, vocab_size, d_model=256, nhead=8, num_layers=4, dropout=0.1):
        super().__init__()
        
        # Keypoint embedding (543 landmarks * 3 coords)
        self.embed = nn.Sequential(
            nn.Linear(543 * 3, d_model * 2),
            nn.LayerNorm(d_model * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 2, d_model),
            nn.LayerNorm(d_model),
        )
        
        self.pos_enc = PositionalEncoding(d_model, dropout=dropout)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            activation='gelu',
            batch_first=True,
            norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        self.classifier = nn.Linear(d_model, vocab_size)
        
        self.d_model = d_model
        self.vocab_size = vocab_size
    
    def forward(self, x, lengths=None):
        # x: (N, T, 543, 3)
        N, T, V, C = x.size()
        x = x.view(N, T, V * C)
        
        x = self.embed(x)
        x = self.pos_enc(x)
        
        # Padding mask
        if lengths is not None:
            mask = torch.arange(T, device=x.device).expand(N, T) >= lengths.unsqueeze(1)
        else:
            mask = None
        
        x = self.transformer(x, src_key_padding_mask=mask)
        logits = self.classifier(x)
        
        return logits

# Create model
vocab = {"<blank>": 0}
for i in range(1000):
    vocab[f"GLOSS_{i}"] = i + 1

model = ISLTranslator(vocab_size=len(vocab), d_model=256, num_layers=4)
model = model.cuda()

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

## 5. Training Loop

In [ ]:
from torch.cuda.amp import autocast, GradScaler
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

# Training config
EPOCHS = 100
BATCH_SIZE = 8
LR = 1e-4
GRAD_ACCUM = 4

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
scaler = GradScaler()
ctc_loss = nn.CTCLoss(blank=0, reduction='mean', zero_infinity=True)

def train_step(batch):
    keypoints = batch['keypoints'].cuda()  # (N, T, 543, 3)
    lengths = batch['lengths'].cuda()      # (N,)
    targets = batch['targets'].cuda()       # (N, S)
    target_lengths = batch['target_lengths'].cuda()
    
    with autocast():
        logits = model(keypoints, lengths)  # (N, T, vocab)
        log_probs = F.log_softmax(logits, dim=-1).permute(1, 0, 2)  # (T, N, vocab)
        loss = ctc_loss(log_probs, targets, lengths, target_lengths)
    
    return loss

print("Training functions ready!")

In [ ]:
# Simple training loop (replace with real data loading)
model.train()

for epoch in range(5):  # Demo with 5 epochs
    total_loss = 0
    
    # Dummy batch for demo
    for step in range(10):
        batch = {
            'keypoints': torch.randn(BATCH_SIZE, 50, 543, 3).cuda(),
            'lengths': torch.full((BATCH_SIZE,), 50).cuda(),
            'targets': torch.randint(1, 100, (BATCH_SIZE, 10)).cuda(),
            'target_lengths': torch.full((BATCH_SIZE,), 10).cuda(),
        }
        
        loss = train_step(batch) / GRAD_ACCUM
        
        scaler.scale(loss).backward()
        
        if (step + 1) % GRAD_ACCUM == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
        
        total_loss += loss.item() * GRAD_ACCUM
    
    scheduler.step()
    print(f"Epoch {epoch+1}, Loss: {total_loss/10:.4f}, LR: {scheduler.get_last_lr()[0]:.2e}")

## 6. Save Model

In [ ]:
# Save checkpoint
CHECKPOINT_DIR = f"{PROJECT_DIR}/checkpoints"
!mkdir -p {CHECKPOINT_DIR}

checkpoint = {
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'vocab': vocab,
}

torch.save(checkpoint, f"{CHECKPOINT_DIR}/model.pt")
print(f"Model saved to {CHECKPOINT_DIR}/model.pt")

## 7. Inference Demo

In [ ]:
# Load and test
model.eval()

# Dummy inference
with torch.no_grad():
    test_kp = torch.randn(1, 100, 543, 3).cuda()
    test_len = torch.tensor([100]).cuda()
    
    logits = model(test_kp, test_len)
    predictions = logits.argmax(dim=-1)
    
    # Decode CTC
    decoded = []
    prev = 0
    for p in predictions[0].cpu().tolist():
        if p != 0 and p != prev:
            decoded.append(p)
        prev = p
    
    print(f"Decoded indices: {decoded[:20]}...")

---
## Next Steps

1. **Download real data**: Update Kaggle dataset name and download ISL-CSLTR
2. **Create annotations**: Map video IDs to gloss sequences
3. **Full training**: Run for 50-100 epochs with real data
4. **Evaluate**: Compute WER on test set
5. **Add translation**: Integrate IndicTrans2 for gloss→Hindi/English